## Prepare your environment

1. **Runtime and compute:** select a Snowflake Notebook on GPU Container Runtime 2.9 with Python 3.12, backed by a GPU compute pool. The query warehouse runs SQL; increasing its size does not increase GPU memory. Choose a pool with enough GPU memory and CPU RAM for the model and graph; there is no validated one-size-fits-all GPU size for this sample.
2. **Local working disk:** provide at least four times the materialized graph-index size in available scratch storage, plus room for retained runs. This is starting guidance, not a measured peak or four times (recommended is 6 times) compressed source-data size. Confirm that `SCRATCH_DIR` uses the intended supported local disk; the notebook prints available space and actual run usage.
3. **Access and permissions:** package installation needs an approved PyPI route, such as a Snowflake external access integration (EAI). The role needs the privileges below; a successful import does not prove them:

| Action | Required access |
| --- | --- |
| Run notebook | `USAGE` on the GPU compute pool and query warehouse; access to the notebook |
| Install | PyPI access through an approved EAI and `USAGE` on that EAI, where required |
| Read data | `IMPORTED PRIVILEGES` on shared database `SNOWFLAKE_SAMPLE_DATA` (or equivalent inherited access) and query access to `TPCH_SF1` |
| Create views | Permission to create the database/schema and create or replace these views; otherwise have an administrator prepare them and skip the view-setup cell |
| Publish a model | `CREATE MODEL` on the destination schema (or schema ownership) |
| Save predictions | Create or overwrite the dedicated output table in that schema |

This notebook uses four TPC-H tables: `CUSTOMER`, `PART`, `ORDERS`, and `LINEITEM`. These are accessed through views created in your configured database/schema over the read-only `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1` data. The LINEITEM view adds a synthetic single-column primary key (`LINEITEM_ID`) since TPC-H uses a composite key (`L_ORDERKEY`, `L_LINENUMBER`).

**Quickstart sample:** by default, the views select 2,000 customers from TPC-H SF1, their orders and line items, and the parts referenced by those line items. This preserves the graph's customer–order–line-item–part relationships. Increase `SAMPLE_CUSTOMERS` only after the small run succeeds.

See [Snowflake GPU runtime 2.9](https://docs.snowflake.com/en/developer-guide/snowflake-ml/container-runtime/releases/gpu/2_9) and [Notebook setup privileges](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-setup) for current platform details.

**Scratch** is local working storage for snapshots, generated rows, model files, predictions, and diagnostics. It may survive a kernel restart if the filesystem remains, but container or machine loss can remove it. Model publication preserves the reusable model; other scratch files remain temporary.

**Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use**

## 1. Enter your configuration

Fill in your warehouse, database, and schema below. Use ordinary unquoted database and schema identifiers. The view-setup cell creates views in your configured database over the read-only `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1`. Views in an owned database are required because Kumo's snapshot uses staged reads, which Snowflake does not allow on shared/imported databases.

This workflow trains and publishes a model, then generates predictions and writes them to the configured Snowflake output table.

January 1, 1998 is the configured prediction date. TPC-H SF1 data spans approximately 1992-01-02 to 1998-08-02. After a kernel restart, rerun this configuration and skip installation.

In [ ]:
from datetime import datetime

WAREHOUSE_NAME = "<YOUR_WAREHOUSE>"
DATABASE_NAME = "<YOUR_DATABASE>"  # Created if it does not exist.
SCHEMA_NAME = "<YOUR_SCHEMA>"  # Source views are created here.

MODEL_NAME = "<YOUR_MODEL_NAME>"  # Model registry name; each fit adds a version.
PREDICTION_TIME = datetime(1998, 1, 1)  # TPC-H data spans ~1992-01-02 to 1998-08-02.
OUTPUT_SCHEMA = "<YOUR_OUTPUT_SCHEMA>"  # Created if it does not exist.
PREDICTION_OUTPUT_TABLE = "<YOUR_PREDICTION_TABLE>"
TPCH_SCHEMA = "TPCH_SF1"
SAMPLE_CUSTOMERS = 2000  # Small, relationship-preserving quickstart slice.

## 1b. Create views over TPC-H sample data

`SNOWFLAKE_SAMPLE_DATA` is a shared database that does not allow staged exports. This cell creates views in your configured database over a bounded, relationship-preserving slice of four TPC-H SF1 tables so Kumo can unload them to local Parquet. The LINEITEM view adds a synthetic primary key (`LINEITEM_ID`) since TPC-H uses a composite key.

Run this once per environment. Safe to rerun (uses `IF NOT EXISTS` and `OR REPLACE`). The printed row counts describe this run's sample; views do not copy source data.

In [ ]:
from snowflake.snowpark.context import get_active_session
from time import perf_counter

view_setup_started = perf_counter()
session = get_active_session()

# Create writable database with views over read-only SNOWFLAKE_SAMPLE_DATA.
# Staged reads require an owned database; SNOWFLAKE_SAMPLE_DATA is shared and read-only.
session.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {DATABASE_NAME}.{SCHEMA_NAME}").collect()

session.sql(f"""
CREATE OR REPLACE VIEW {DATABASE_NAME}.{SCHEMA_NAME}.CUSTOMER AS
SELECT C_CUSTKEY, C_NAME, C_NATIONKEY, C_MKTSEGMENT, C_ACCTBAL
FROM SNOWFLAKE_SAMPLE_DATA.{TPCH_SCHEMA}.CUSTOMER
WHERE C_CUSTKEY <= {SAMPLE_CUSTOMERS}
""").collect()

session.sql(f"""
CREATE OR REPLACE VIEW {DATABASE_NAME}.{SCHEMA_NAME}.ORDERS AS
SELECT O_ORDERKEY, O_CUSTKEY, O_ORDERDATE, O_ORDERSTATUS, O_ORDERPRIORITY, O_TOTALPRICE, O_SHIPPRIORITY
FROM SNOWFLAKE_SAMPLE_DATA.{TPCH_SCHEMA}.ORDERS
WHERE O_CUSTKEY <= {SAMPLE_CUSTOMERS}
""").collect()

# LINEITEM needs a synthetic single-column primary key (TPC-H uses composite L_ORDERKEY + L_LINENUMBER).
session.sql(f"""
CREATE OR REPLACE VIEW {DATABASE_NAME}.{SCHEMA_NAME}.LINEITEM AS
SELECT
    (L_ORDERKEY * 100 + L_LINENUMBER) AS LINEITEM_ID,
    L_ORDERKEY, L_PARTKEY, L_LINENUMBER, L_QUANTITY,
    L_EXTENDEDPRICE, L_DISCOUNT, L_TAX, L_SHIPDATE,
    L_RETURNFLAG, L_LINESTATUS, L_SHIPMODE
FROM SNOWFLAKE_SAMPLE_DATA.{TPCH_SCHEMA}.LINEITEM
WHERE L_ORDERKEY IN (
    SELECT O_ORDERKEY FROM {DATABASE_NAME}.{SCHEMA_NAME}.ORDERS
)
""").collect()

session.sql(f"""
CREATE OR REPLACE VIEW {DATABASE_NAME}.{SCHEMA_NAME}.PART AS
SELECT P_PARTKEY, P_NAME, P_MFGR, P_BRAND, P_TYPE, P_SIZE, P_CONTAINER, P_RETAILPRICE
FROM SNOWFLAKE_SAMPLE_DATA.{TPCH_SCHEMA}.PART
WHERE P_PARTKEY IN (
    SELECT L_PARTKEY FROM {DATABASE_NAME}.{SCHEMA_NAME}.LINEITEM
)
""").collect()

for table_name in ("CUSTOMER", "ORDERS", "LINEITEM", "PART"):
    count = session.sql(
        f"SELECT COUNT(*) FROM {DATABASE_NAME}.{SCHEMA_NAME}.{table_name}"
    ).collect()[0][0]
    print(f"{table_name} rows: {count:,}")
print(f"View setup and row counts: {perf_counter() - view_setup_started:.1f} seconds")

## 2. Install Kumo Engine from PyPI

Run this installation cell once. **Then restart the kernel, rerun configuration, skip installation, and continue at imports.** Do not use Run All across the restart boundary. Installation uses the active interpreter.

In [ ]:
%pip install kumo-engine[snowflake]

<a id="runtime"></a>
## Import and configure scratch

After each kernel restart, run configuration and these cells. They do not depend on variables created during installation. Configure scratch before constructing any Kumo table, graph, or trainer.

In [ ]:
import sys
import shutil
import tempfile
from importlib.metadata import version
from pathlib import Path

import kumo.engine as kumo
from kumo.engine import SnowflakeConnector
from kumoapi.graph import Edge
from kumoapi.jobs import PredictionArtifactType, WriteMode

In [ ]:
SCRATCH_DIR = "/tmp/kumoai-local"
scratch = Path(SCRATCH_DIR)
scratch.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=scratch) as probe:
    probe.write(b"kumo scratch check")
    probe.flush()
kumo.configure(scratch_dir=SCRATCH_DIR)
print("Kumo version:", kumo.__version__)
print("Free working space (GiB):", round(shutil.disk_usage(scratch).free / 2**30, 1))

The printed version identifies the installed build. The probe checks write access and free space at this path; it does not establish that the path is on supported local storage. Confirm the backing disk and capacity before materializing data.

## Connect and inspect the source

`from_spcs_env()` uses runtime-provided account and host settings and the mounted OAuth token. Never copy credentials into this notebook. A single connector reads and writes through your configured database/schema. Confirm all four table views appear.

In [ ]:
connector = kumo.SnowflakeConnector.from_spcs_env(
    warehouse=WAREHOUSE_NAME, database=DATABASE_NAME, schema_name=SCHEMA_NAME,
)
print("Tables:", connector.table_names())

## Describe the four tables

A primary key identifies a row. A physical type describes its stored representation, such as a date; a semantic type describes its modeling role, such as a category or numerical value.

In [ ]:
customer = (
    kumo.Table.from_source_table(
        connector["CUSTOMER"],
        ["C_CUSTKEY", "C_NATIONKEY", "C_MKTSEGMENT", "C_ACCTBAL"],
        primary_key="C_CUSTKEY",
    )
)

part = (
    kumo.Table.from_source_table(
        connector["PART"],
        ["P_PARTKEY", "P_NAME", "P_BRAND", "P_TYPE",
         "P_SIZE", "P_CONTAINER", "P_RETAILPRICE"],
        primary_key="P_PARTKEY",
    )
)

Event timestamps let Kumo separate information available before a prediction boundary from future outcomes. Using future information as an input causes temporal leakage.

In [ ]:
orders = (
    kumo.Table.from_source_table(
        connector["ORDERS"],
        ["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_ORDERSTATUS",
         "O_ORDERPRIORITY", "O_TOTALPRICE"],
        primary_key="O_ORDERKEY",
        time_column="O_ORDERDATE",
    )
)

In [ ]:
lineitem = (
    kumo.Table.from_source_table(
        connector["LINEITEM"],
        ["LINEITEM_ID", "L_ORDERKEY", "L_PARTKEY", "L_SHIPDATE",
         "L_QUANTITY", "L_EXTENDEDPRICE", "L_DISCOUNT"],
        primary_key="LINEITEM_ID",
        time_column="L_SHIPDATE",
    )
)

## Connect the tables and materialize a snapshot

A **graph** describes the tables and their relationships. Each arrow below goes from a foreign-key column to the destination table's primary key:

```text
LINEITEM.L_PARTKEY  -> PART.P_PARTKEY
LINEITEM.L_ORDERKEY -> ORDERS.O_ORDERKEY
ORDERS.O_CUSTKEY    -> CUSTOMER.C_CUSTKEY
```

Validation checks the graph definition; it is not a blanket guarantee of source-data quality. A **snapshot** materializes source data into the local graph index used for training and prediction. Snapshot creation reads the tables and consumes local disk. Reuse requires an unchanged, valid retained snapshot; loading a model does not restore a lost index.

In [ ]:
from IPython.display import display

graph = kumo.Graph(
    tables=[customer, part, orders, lineitem],
    edges=[
        Edge(src_table="ORDERS", fkey="O_CUSTKEY", dst_table="CUSTOMER"),
        Edge(src_table="LINEITEM", fkey="L_ORDERKEY", dst_table="ORDERS"),
        Edge(src_table="LINEITEM", fkey="L_PARTKEY", dst_table="PART"),
    ],
)

graph.validate()
display(graph.visualize().as_widget())

snapshot = graph.snapshot()

print("Snapshot:", snapshot.id)

## Define the prediction question

Predictive Query Language (PQL) is Kumo Engine's declarative syntax for defining a predictive modeling task: it specifies the target, the entity to score, and the data context. This default query predict how many orders a customer active in the past year will place in the next 90 days.

In [ ]:
# Regression: predict how many orders a customer active in the past year will place in the next 90 days.
pquery = kumo.PredictiveQuery(
    graph=graph,
    query=(
        "PREDICT COUNT(ORDERS.*, 0, 90, days) "
        "FOR EACH CUSTOMER.C_CUSTKEY "
        "WHERE COUNT(ORDERS.*, -365, 0, days) > 0"
    ),
)
pquery.validate()
print("Task type:", pquery.get_task_type())

### Optional task alternatives

The next two code cells are commented out. To try one, skip or comment out the binary query cell above, uncomment exactly one alternative, and run it before training. 

In [ ]:
# Binary classification: predict whether a customer active in the past year will order again in the next 90 days.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query="""
#         PREDICT COUNT(ORDERS.*, 0, 90, days) > 0
#         FOR EACH CUSTOMER.C_CUSTKEY
#         WHERE COUNT(ORDERS.*, -365, 0, days) > 0
#     """,
# )

# pquery.validate()
# print("Task type:", pquery.get_task_type())

In [ ]:
# Link prediction: rank up to five parts that may ship in the next 90 days for an order with recent line items.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         "PREDICT LIST_DISTINCT(LINEITEM.L_PARTKEY, 0, 90, days) "
#         "RANK TOP 5 FOR EACH ORDERS.O_ORDERKEY "
#         "WHERE COUNT(LINEITEM.*, -365, 0, days) > 0"
#     ),
# )
# pquery.validate()
# print("Task type:", pquery.get_task_type())

### Generate a Training Table

After your query is validated, generate a training table to use for model fitting. 

In [ ]:
train_table = pquery.generate_training_table(snapshot)

## Create a Trainer

The suggested model plan controls architecture, sampling, and optimization. The model plan is fully customizable; inspect and adjust its settings and validation selection metric for your task before creating `trainer = kumo.Trainer(model_plan)`. Compare like-for-like validation metrics and reserve test metrics for evaluating the selected model.

In [ ]:
model_plan = pquery.suggest_model_plan(snapshot)

trainer = kumo.Trainer(model_plan)

### Train a Model

After creating a Trainer from the model plan, call `fit()` with the training table and compatible graph snapshot. The fit step also publishes the model to Snowflake.

For a deliberately local-only experiment, set `model_output = None` in this cell before running it.

In [ ]:
model_output = kumo.ModelOutputConfig(
    connector, location=kumo.ModelRegistryLocation(MODEL_NAME),
)

training_result = trainer.fit(
    train_table, snapshot=snapshot, export_config=model_output,
)

### Inspect the selected result

Review the reported metrics and their stage names. Available metrics vary by task and build; this example does not promise calibration output or a fixed set. A successful fit alone does not establish business usefulness.

The reusable bundle and registry publication preserve model state. They do not archive the complete experiment history. Continue to batch prediction below.

In [ ]:
print("Available metrics:", training_result.metrics)
print("Reusable bundle:", training_result.bundle_dir)

### Generate the scoring population

For this temporal query, use a timezone-naive timestamp interpreted as UTC. Reuse the fitted trainer's saved training-table generation plan. The generated population contains customers meeting the past-year restriction at that date. This timestamp example does not apply to static queries.

In [ ]:
prediction_table = pquery.generate_prediction_table(
    snapshot,
    prediction_time=PREDICTION_TIME,
    plan=trainer.training_table_generation_plan,
)

<a id="prediction"></a>
## Generate Batch Predictions

After training a model, generate batch predictions and write them to the configured Snowflake table.

Check the source date range against `PREDICTION_TIME` before scoring. A chosen date alone does not guarantee the following 90 days of observed outcomes for historical evaluation.

In [ ]:
# The output schema must exist before Kumo writes to it.
session.sql(f"CREATE SCHEMA IF NOT EXISTS {DATABASE_NAME}.{OUTPUT_SCHEMA}").collect()

# Separate connector for writes, reusing the source connector's identity and warehouse.
output_connector = SnowflakeConnector(
    identity=connector.identity,
    warehouse=connector.warehouse,
    database=connector.database,
    role=connector.role,
    schema_name=OUTPUT_SCHEMA,
)

# Specific threshold or candidate count are required for binary classification and link prediction tasks respectively. 
# prediction_options = {}
# task_type = pquery.get_task_type()
# if task_type == TaskType.BINARY_CLASSIFICATION:
#     prediction_options["binary_classification_threshold"] = 0.5
# elif task_type.is_link_pred:
#     prediction_options["num_classes_to_return"] = 10

prediction_output = kumo.PredictionOutputConfig(
    connector=output_connector,
    artifact_type=PredictionArtifactType.PREDICTIONS,
    table_name=PREDICTION_OUTPUT_TABLE,
    write_mode=WriteMode.OVERWRITE,
)

prediction_result = trainer.predict(
    prediction_table, snapshot=snapshot, output_config=prediction_output,
)
print("Rows:", prediction_result.num_rows)
print("Local Parquet files:", prediction_result.files)
print("Snowflake output:", f"{DATABASE_NAME}.{OUTPUT_SCHEMA}.{PREDICTION_OUTPUT_TABLE}")